# Lab 07: Diffusion Visualization

**Module 07** | Read `notes/07-diffusion-theory.pdf` before running this notebook.

Visualize the forward noising process and a linear beta schedule on MNIST digits, the foundation for DDPM training.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Lab 07: Forward diffusion on MNIST

**Diffusion models** learn to undo gradual noise corruption. Before training a denoiser (Lab 08), this lab visualizes the **forward process**: how a clean MNIST digit gets noisier step by step until it looks like random static.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Forward diffusion** | A fixed recipe that adds a little Gaussian noise at each timestep, slowly destroying image structure. |
| **Timestep (t)** | An integer step counter from 1 to T. Higher t means more noise has been added. |
| **Beta schedule (beta_t)** | A list of small positive numbers, one per timestep, controlling how much noise is added at each step. |
| **Alpha (alpha_t)** | Defined as `1 - beta_t`. It is the fraction of the previous image kept at step t. |
| **Alpha_bar (cumulative alpha)** | The product of all alphas up to t. Lets us jump from clean image x_0 to noisy x_t in one formula. |
| **Gaussian noise** | Random numbers drawn from a bell curve (mean 0, variance 1). Standard choice for diffusion. |
| **Signal-to-noise ratio** | How much original image content remains vs how much random noise dominates. |
| **Markov process** | Each step depends only on the previous step, not the full history. Simplifies the math. |
| **Closed-form noising** | A single equation to compute x_t directly from x_0 without looping t times. |
| **Variance schedule** | Another name for the beta schedule: it schedules how noise variance grows over time. |

Refer back to this table whenever a term appears in code or output.


### Concept: Forward diffusion and the beta schedule

**Forward diffusion** is the corruption phase. Imagine filming a digit while someone slowly turns up TV static. At timestep `t=0` you see the original digit. At `t=500` you see almost pure noise.

**Why add noise at all?** Training needs examples at every noise level. The network in Lab 08 will learn: "given this blurry/noisy image at step t, guess what noise was added." At generation time we start from pure noise and walk backward.

**Beta schedule:** We pick `beta_t` values that grow slowly from a tiny start (0.0001) to a larger end (0.02). Small early betas mean early steps change the image only slightly. Later betas add stronger corruption.

**Alpha_bar:** Define `alpha_t = 1 - beta_t`. Then `alpha_bar_t = alpha_1 * alpha_2 * ... * alpha_t`. When `alpha_bar_t` is close to 1, the image is almost clean. When `alpha_bar_t` is near 0, noise dominates.

**Closed-form formula** (same noise epsilon for fair comparison across timesteps):

`x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon`

Think of it as a weighted blend: sqrt(alpha_bar_t) times the clean image plus sqrt(1 - alpha_bar_t) times random noise.


### Step 1: Build the beta schedule

**What this does:** Creates 500 timesteps of linear betas, derives alphas and alpha_bars, prints schedule statistics, and plots beta_t and alpha_bar_t curves.

**Why we do this:** The schedule controls how fast information is destroyed. These plots let you see whether corruption is gradual (good) or too abrupt.


**What to look for in the output**

T should be 500. beta_1 near 0.0001, beta_T near 0.02. alpha_bar_T should be a very small number (nearly zero), meaning almost no signal left. The right plot should decay smoothly toward 0.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
T = 500  # total forward diffusion steps

# Linear schedule: noise per step grows evenly from start to end
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
# Cumulative product: alpha_bar_t = how much original signal survives by step t
alpha_bars = torch.cumprod(alphas, dim=0)

print("Diffusion schedule statistics")
print(f"  T (timesteps):     {T}")
print(f"  beta_1:            {betas[0]:.6f}")
print(f"  beta_T:            {betas[-1]:.4f}")
print(f"  alpha_bar_1:       {alpha_bars[0]:.4f}")
print(f"  alpha_bar_T:       {alpha_bars[-1]:.6f}")
print(f"  Signal at t=100:   {alpha_bars[99]:.4f} (fraction of x_0 variance retained)")
print(f"  Signal at t=250:   {alpha_bars[249]:.4f}")
print(f"  Signal at t=500:   {alpha_bars[499]:.6f} (nearly pure noise)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(betas.numpy())
axes[0].set_title("Linear beta schedule")
axes[0].set_xlabel("Timestep t")
axes[0].set_ylabel("beta_t")
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_bars.numpy())
axes[1].set_title("Cumulative alpha_bar = prod(1 - beta)")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("alpha_bar_t")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 2: Load one MNIST digit

**What this does:** Loads MNIST, picks the first training image, and prints its label and pixel statistics.

**Why we do this:** A single fixed digit makes it easy to compare how the same structure dissolves under increasing noise.


**What to look for in the output**

Digit label is an integer 0-9 (index 0 is often digit 5 in MNIST). Shape `(1, 1, 28, 28)` after adding batch dimension. Pixel mean/std should be modest after normalization.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
x0, label = train_set[0]  # x0 = clean image, label = digit class
x0 = x0.unsqueeze(0).to(device)  # add batch dimension, move to device

print("Selected MNIST example")
print(f"  Dataset index: 0")
print(f"  Digit label:   {label}")
print(f"  Shape:         {tuple(x0.shape)}")
print(f"  Pixel mean:    {x0.mean():.4f}")
print(f"  Pixel std:     {x0.std():.4f}")


### Step 3: Apply forward diffusion at selected timesteps

**What this does:** Corrupts the digit at timesteps [0, 50, 100, 200, 500] using one fixed noise tensor, prints a stats table, and shows a row of images.

**Why we do this:** You see the forward process with your own eyes. Early t looks almost clean; late t is unrecognizable. The table links alpha_bar to mean/std drift.


**What to look for in the output**

At t=0, alpha_bar is 1.0 and the image is crisp. By t=500, mean/std drift toward 0 and 1 (noise-like). The printed interpretation column should match what you see.


In [ ]:
timesteps = [0, 50, 100, 200, 500]
noisy_images = []

# Fixed seed so the SAME noise is reused at every t (fair visual comparison)
torch.manual_seed(42)
noise = torch.randn_like(x0)

print(f"{'t':>4} | {'alpha_bar':>10} | {'mean':>8} | {'std':>8} | interpretation")
print("-" * 70)

for t in timesteps:
    if t == 0:
        xt = x0
        ab = torch.tensor(1.0)
        note = "clean image (no noise added)"
    else:
        ab = alpha_bars[t - 1]  # index t-1 because alpha_bars is 0-based
        ab_dev = ab.to(device)
        # Closed-form forward diffusion: blend clean image and noise
        xt = torch.sqrt(ab_dev) * x0 + torch.sqrt(1.0 - ab_dev) * noise
        if t <= 100:
            note = "mostly signal, slight blur"
        elif t <= 200:
            note = "mixed signal and noise"
        else:
            note = "dominated by noise"
    stats = xt.detach().cpu()
    print(
        f"{t:4d} | {ab.item():10.6f} | {stats.mean():8.4f} | {stats.std():8.4f} | {note}"
    )
    noisy_images.append(stats)

# Display: map [-1,1] normalized pixels back to [0,1] for imshow
fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 3))
for ax, t, img in zip(axes, timesteps, noisy_images):
    disp = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(disp, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t = {t}")
    ax.axis("off")
plt.suptitle(f"Forward diffusion of MNIST digit {label}")
plt.tight_layout()
plt.show()


### Step 4: Interpret noise level

When `alpha_bar_t` is close to 1, `x_t` is almost identical to `x_0`. When `alpha_bar_t` approaches 0, the sqrt terms weight noise heavily and the digit becomes unrecognizable.

DDPM training (Lab 08) samples random `t` values across this entire range so the network learns denoising at every signal-to-noise ratio. The forward process you just visualized is the **training data generator**: it creates noisy inputs paired with the noise that was added.
